In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/metadata.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv


In [2]:
# ==============================
# Numerical Libraries
# ==============================

import numpy as np
import pandas as pd

# ==============================
# Visualization Libraries
# ==============================

import matplotlib.pyplot as plt
import seaborn as sns

# Better plot style
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

# ==============================
# Ignore warnings
# ==============================

import warnings
warnings.filterwarnings("ignore")

# ==============================
# Machine Learning
# ==============================

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    KFold,
    cross_val_score
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    OrdinalEncoder
)

# Regression Models
from sklearn.linear_model import LinearRegression

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Boosting Libraries
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [3]:
train = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")

test = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv")

metadata = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/metadata.csv")

sample_submission = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv")

In [4]:
train.shape

(138701, 50)

In [5]:
train.head()

,TransactionID,TargetValue,AssetID,ProductConfigID,DataOriginCode,VendorPartnerID,ManufactureYear,OperationalHoursMeter,UtilizationTier,TransactionDate,...,col20,col21,col22,col23,col24,col25,col27,col28,col29,col30
0,1139309,57000.0,117665,88,ACH138,GTX,1997,4640.0,Low,2005-03-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Standard,Conventional
1,1139316,26500.0,1001282,4616,ACH138,GTX,2005,508.0,Low,2009-12-18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1139317,21000.0,772709,1948,ACH138,GTX,1994,11540.0,High,2005-08-26,...,Steel,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN
3,1139322,27000.0,902010,3550,ACH138,GTX,2002,4883.0,High,2006-11-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1139333,21500.0,1036259,36014,ACH138,GTX,2009,302.0,Low,2010-08-27,...,Rubber,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138701 entries, 0 to 138700
Data columns (total 50 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   TransactionID              138701 non-null  int64  
 1   TargetValue                138701 non-null  float64
 2   AssetID                    138701 non-null  int64  
 3   ProductConfigID            138701 non-null  int64  
 4   DataOriginCode             138701 non-null  object 
 5   VendorPartnerID            138701 non-null  object 
 6   ManufactureYear            138701 non-null  int64  
 7   OperationalHoursMeter      82058 non-null   float64
 8   UtilizationTier            50475 non-null   object 
 9   TransactionDate            138701 non-null  object 
 10  Spec_FullDescriptor        138701 non-null  object 
 11  Spec_BaseClass             138701 non-null  object 
 12  Spec_SubClass              101716 non-null  object 
 13  Spec_ReleaseSeries         32

In [7]:
train.describe()

,TransactionID,TargetValue,AssetID,ProductConfigID,ManufactureYear,OperationalHoursMeter
count,1.387010e+05,138701.000000,1.387010e+05,138701.000000,138701.000000,8.205800e+04
mean,2.186276e+06,41521.874047,1.204814e+06,7286.421922,1891.640853,4.553719e+03
std,1.128497e+06,26361.125948,5.287376e+05,7542.421537,306.992001,2.792204e+04
min,1.139309e+06,7500.000000,8.000000e+00,39.000000,1001.000000,0.000000e+00
25%,1.465754e+06,20500.000000,1.007930e+06,1693.000000,1990.000000,0.000000e+00
50%,1.751884e+06,35000.000000,1.246956e+06,3852.000000,1998.000000,1.620000e+03
75%,2.372139e+06,57000.000000,1.495817e+06,11873.000000,2004.000000,5.001000e+03
max,6.333410e+06,142000.000000,2.478476e+06,37209.000000,2015.000000,1.729600e+06


In [8]:
train.isnull().sum()

TransactionID                     0
TargetValue                       0
AssetID                           0
ProductConfigID                   0
DataOriginCode                    0
VendorPartnerID                   0
ManufactureYear                   0
OperationalHoursMeter         56643
UtilizationTier               88226
TransactionDate                   0
Spec_FullDescriptor               0
Spec_BaseClass                    0
Spec_SubClass                 36985
Spec_ReleaseSeries           105805
Spec_VariantModifier          86369
AssetScaleFactor              41210
FunctionalClassification          0
RegionCode                        0
InventoryGroupCategory            0
InventoryGroupDescription         0
col1                         104915
CabinType                         0
Forks                        109382
col3                         109448
col4                         128320
DrivetrainType                89956
col5                         128320
col6                        

In [9]:
for df in [train, test]:

    df["TransactionDate"] = pd.to_datetime(df["TransactionDate"])

    df["TransactionYear"] = df["TransactionDate"].dt.year
    df["TransactionMonth"] = df["TransactionDate"].dt.month
    df["TransactionQuarter"] = df["TransactionDate"].dt.quarter
    df["TransactionWeekday"] = df["TransactionDate"].dt.weekday

    df["AssetAge"] = (
        df["TransactionYear"] - df["ManufactureYear"]
    )

    df["AgeSquared"] = df["AssetAge"] ** 2

    df["DescriptorLength"] = (
        df["Spec_FullDescriptor"]
        .fillna("")
        .str.len()
    )

    df["WordCount"] = (
        df["Spec_FullDescriptor"]
        .fillna("")
        .str.split()
        .str.len()
    )

    df["DigitCount"] = (
        df["Spec_FullDescriptor"]
        .fillna("")
        .str.count(r"\d")
    )

    df["HasOperationalHours"] = (
        df["OperationalHoursMeter"]
        .notna()
        .astype(int)
    )

    df["MissingCount"] = df.isna().sum(axis=1)

In [10]:
TARGET = "TargetValue"
ID = "TransactionID"

X = train.drop(columns=[TARGET])
y = train[TARGET]

X_test = test.copy()



# Drop original datetime column
X = X.drop(columns=["TransactionDate"])
X_test = X_test.drop(columns=["TransactionDate"])

In [11]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

numerical_cols = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

if ID in numerical_cols:
    numerical_cols.remove(ID)



num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

X[numerical_cols] = num_imputer.fit_transform(X[numerical_cols])
X_test[numerical_cols] = num_imputer.transform(X_test[numerical_cols])

X[categorical_cols] = cat_imputer.fit_transform(X[categorical_cols])
X_test[categorical_cols] = cat_imputer.transform(X_test[categorical_cols])

In [12]:
from sklearn.preprocessing import LabelEncoder

for col in categorical_cols:

    encoder = LabelEncoder()

    combined = pd.concat(
        [X[col], X_test[col]],
        axis=0
    ).astype(str)

    encoder.fit(combined)

    X[col] = encoder.transform(X[col].astype(str))
    X_test[col] = encoder.transform(X_test[col].astype(str))

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
y_train_log = np.log1p(y_train)
y_valid_log = np.log1p(y_valid)

In [14]:
model = LGBMRegressor(
    objective="regression",
    n_estimators=5000,
    learning_rate=0.015,
    num_leaves=255,
    max_depth=-1,
    min_child_samples=10,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.3,
    reg_lambda=5,
    random_state=42,
    n_jobs=-1
)

In [15]:
model.fit(
    X_train,
    y_train_log
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.021548 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2452
[LightGBM] [Info] Number of data points in the train set: 110960, number of used features: 57
[LightGBM] [Info] Start training from score 10.424734


LGBMRegressor(colsample_bytree=0.9, learning_rate=0.015, min_child_samples=10,
              n_estimators=5000, n_jobs=-1, num_leaves=255,
              objective='regression', random_state=42, reg_alpha=0.3,
              reg_lambda=5, subsample=0.9)

In [16]:
from sklearn.metrics import root_mean_squared_log_error

valid_pred_log = model.predict(X_valid)

valid_pred = np.expm1(valid_pred_log)

score = root_mean_squared_log_error(
    y_valid,
    valid_pred
)

print(score)

0.20260525081125316


In [17]:
y_log = np.log1p(y)

final_model = LGBMRegressor(
    objective="regression",
    n_estimators=5000,
    learning_rate=0.015,
    num_leaves=255,
    max_depth=-1,
    min_child_samples=10,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.3,
    reg_lambda=5,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X, y_log)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025976 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2472
[LightGBM] [Info] Number of data points in the train set: 138701, number of used features: 57
[LightGBM] [Info] Start training from score 10.424111


LGBMRegressor(colsample_bytree=0.9, learning_rate=0.015, min_child_samples=10,
              n_estimators=5000, n_jobs=-1, num_leaves=255,
              objective='regression', random_state=42, reg_alpha=0.3,
              reg_lambda=5, subsample=0.9)

In [18]:
test_pred_log = final_model.predict(X_test)

test_pred = np.expm1(test_pred_log)

In [19]:
submission = pd.DataFrame({
    "TransactionID": test["TransactionID"],
    "TargetValue": test_pred
})

submission.to_csv("submission.csv", index=False)

submission.head()

,TransactionID,TargetValue
0,1139307,68011.991105
1,1139419,80537.571675
2,1139482,32862.084778
3,1139522,19849.144339
4,1139684,12701.423873
